# Research Agent RL — Week 3: REINFORCE Policy Training

**Setup:** GPU T4 x1, Internet ON

This notebook:
1. Clones the repo and installs dependencies
2. Downloads the Week 1 SFT checkpoint (frozen)
3. Builds the HotpotQA tool index
4. Trains a lightweight MLP stopping policy with REINFORCE
5. Evaluates the RL policy vs. baseline policies
6. Saves the trained policy and results

## 1. Check GPU

In [ ]:
import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else 'No GPU — enable T4 in Settings!')

## 2. Clone repo & install

In [ ]:
import os

REPO_URL = 'https://github.com/202520030411/Research_Agent_RL.git'
REPO_DIR = '/kaggle/working/Research_Agent_RL'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())

In [ ]:
!pip install -q -r requirements.txt

## 3. Restore Week 1 SFT checkpoint

In [ ]:
import os, shutil, glob

ADAPTER_DIR = 'checkpoints/qwen-sft/final'

if not os.path.exists(ADAPTER_DIR):
    print('Downloading Week 1 checkpoint from kernel output...')
    !mkdir -p /tmp/w1_out
    !kaggle kernels output wuyue22/rl-sft-research -p /tmp/w1_out

    src = '/tmp/w1_out/Research_Agent_RL/checkpoints/qwen-sft/final'
    if os.path.exists(src):
        os.makedirs('checkpoints/qwen-sft', exist_ok=True)
        shutil.copytree(src, ADAPTER_DIR)
        print(f'Adapter copied → {ADAPTER_DIR}')
    else:
        ckpts = sorted(glob.glob('/tmp/w1_out/Research_Agent_RL/checkpoints/qwen-sft/checkpoint-*'))
        if ckpts:
            shutil.copytree(ckpts[-1], ADAPTER_DIR)
            print(f'Using latest checkpoint: {ckpts[-1]}')
        else:
            print('[ERROR] No checkpoint found in kernel output.')
else:
    print(f'Checkpoint already present at {ADAPTER_DIR}')

!ls -lh {ADAPTER_DIR}

## 4. Load SFT model (frozen)

In [ ]:
import json, torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

ADAPTER_DIR = 'checkpoints/qwen-sft/final'

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'

with open(f'{ADAPTER_DIR}/adapter_config.json') as f:
    base_model_name = json.load(f).get('base_model_name_or_path', 'Qwen/Qwen2.5-0.5B-Instruct')

print(f'Loading base model: {base_model_name}')
base = AutoModelForCausalLM.from_pretrained(
    base_model_name, device_map={'': 0}, torch_dtype=torch.float16, trust_remote_code=True
)
sft_model = PeftModel.from_pretrained(base, ADAPTER_DIR)
sft_model.eval()
sft_model.config.use_cache = True
for p in sft_model.parameters():
    p.requires_grad_(False)

print(f'SFT model loaded (frozen). VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB')

## 5. Load HotpotQA & build tool index

In [ ]:
from datasets import load_dataset
from agent.tools import ToolExecutor

N_TRAIN_Q = 500    # episodes drawn from train split for REINFORCE
N_VAL_Q   = 100    # questions for evaluation

print('Loading HotpotQA...')
hf = load_dataset('hotpot_qa', 'distractor', trust_remote_code=True)
train_hf = hf['train']
val_hf   = hf['validation']

train_questions = [
    {'question': train_hf[i]['question'], 'answer': train_hf[i]['answer']}
    for i in range(min(N_TRAIN_Q, len(train_hf)))
]
val_questions = [
    {'question': val_hf[i]['question'], 'answer': val_hf[i]['answer']}
    for i in range(min(N_VAL_Q, len(val_hf)))
]
print(f'Train pool: {len(train_questions)} questions | Val pool: {len(val_questions)} questions')

executor = ToolExecutor(top_k=2)
executor.build_from_hotpotqa(val_hf, index_path='data/tool_index.jsonl')
print(f'Tool index: {len(executor)} paragraphs')

## 6. Initialise RL policy network

In [ ]:
from rl.policy_network import PolicyNetwork
from rl.rewards import compute_reward
from rl.rollout import RolloutCollector

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

policy    = PolicyNetwork(hidden=64).to(DEVICE)
optimiser = torch.optim.Adam(policy.parameters(), lr=3e-4)

collector = RolloutCollector(
    sft_model=sft_model,
    tokenizer=tokenizer,
    tool_executor=executor,
    policy_network=policy,
    max_steps=6,
    alpha=0.1,
    beta=0.05,
    epsilon=0.05,
)

print('Policy network:', sum(p.numel() for p in policy.parameters()), 'parameters')

## 7. REINFORCE training loop

In [ ]:
import random, time

# Hyperparameters — keep small for a T4 single-GPU run
N_EPOCHS         = 40    # policy update steps
N_EPISODES_EPOCH = 12    # episodes per update (limited by GPU memory & time)
BASELINE_WINDOW  = 50    # moving average window for reward baseline

random.seed(42)
torch.manual_seed(42)

os.makedirs('checkpoints/rl-policy', exist_ok=True)

running_rewards = []
log = []

for epoch in range(1, N_EPOCHS + 1):
    t0 = time.time()
    policy.train()

    # Sample questions for this epoch
    batch_qs = random.sample(train_questions, min(N_EPISODES_EPOCH, len(train_questions)))
    episodes = collector.collect_batch(batch_qs)

    raw_rewards = [ep.reward.total for ep in episodes]
    running_rewards.extend(raw_rewards)
    baseline = sum(running_rewards[-BASELINE_WINDOW:]) / len(running_rewards[-BASELINE_WINDOW:])

    # REINFORCE loss
    policy_loss  = torch.tensor(0.0, device=DEVICE, requires_grad=True)
    n_transitions = 0

    for ep in episodes:
        R         = torch.tensor(ep.reward.total, device=DEVICE)
        advantage = R - baseline
        for trans in ep.transitions:
            lp = trans.log_prob.to(DEVICE)
            policy_loss  = policy_loss - lp * advantage
            n_transitions += 1

    if n_transitions > 0:
        policy_loss = policy_loss / n_transitions
        optimiser.zero_grad()
        policy_loss.backward()
        torch.nn.utils.clip_grad_norm_(policy.parameters(), 1.0)
        optimiser.step()

    acc        = sum(1 for ep in episodes if ep.result.correct) / len(episodes)
    avg_steps  = sum(ep.result.n_steps for ep in episodes) / len(episodes)
    avg_reward = sum(raw_rewards) / len(raw_rewards)
    elapsed    = time.time() - t0

    entry = {
        'epoch': epoch, 'loss': policy_loss.item(),
        'reward': avg_reward, 'baseline': baseline,
        'accuracy': acc, 'avg_steps': avg_steps, 'elapsed_s': round(elapsed, 1)
    }
    log.append(entry)
    print(f"Epoch {epoch:3d}/{N_EPOCHS}  "
          f"loss={entry['loss']:+.4f}  R={avg_reward:+.3f}  base={baseline:+.3f}  "
          f"acc={acc:.2f}  steps={avg_steps:.1f}  ({elapsed:.0f}s)")

    # Save checkpoint every 10 epochs
    if epoch % 10 == 0:
        ckpt = f'checkpoints/rl-policy/policy_epoch{epoch:03d}.pt'
        policy.save(ckpt)
        print(f'  → Saved {ckpt}')

print('\nTraining complete!')

## 8. Evaluate RL policy vs. baselines

In [ ]:
from tqdm import tqdm
from agent.agent import ResearchAgent
from agent.stopping import FixedStepPolicy, ConfidencePolicy, NeverStop, RLPolicy
from eval.metrics import compute_metrics, compare_policies

policy.eval()

policies_to_eval = {
    'RL Policy':           RLPolicy(policy_network=policy, threshold=0.5),
    'FixedStep (N=3)':     FixedStepPolicy(max_steps=3),
    'Confidence (τ=0.75)': ConfidencePolicy(threshold=0.75),
    'NeverStop':           NeverStop(),
}

eval_results = {}

for name, stopping_policy in policies_to_eval.items():
    print(f'\n--- {name} ---')
    agent = ResearchAgent(sft_model, tokenizer, executor, stopping_policy, max_steps=6)
    results = []
    for q in tqdm(val_questions, desc=name):
        stopping_policy.reset()
        results.append(agent.run(q['question'], gold_answer=q['answer']))
    eval_results[name] = results
    m = compute_metrics(results)
    print(f"  acc={m['accuracy']:.3f}  steps={m['avg_steps']:.1f}  "
          f"tools={m['avg_tool_calls']:.1f}  eff={m['efficiency_score']:.4f}")

In [ ]:
print(compare_policies(eval_results))

## 9. Save results & policy

In [ ]:
import json, os

# Save final policy
final_policy_path = '/kaggle/working/rl_policy_final.pt'
policy.save(final_policy_path)
print(f'Policy saved → {final_policy_path}')

# Save training log
log_path = '/kaggle/working/rl_training_log.json'
with open(log_path, 'w') as f:
    json.dump(log, f, indent=2)
print(f'Training log saved → {log_path}')

# Save evaluation results
serializable = {}
for name, results in eval_results.items():
    serializable[name] = [
        {'question': r.question, 'gold_answer': r.gold_answer,
         'pred_answer': r.pred_answer, 'correct': r.correct,
         'n_steps': r.n_steps, 'n_tool_calls': r.n_tool_calls,
         'stopped_by': r.stopped_by}
        for r in results
    ]

results_path = '/kaggle/working/rl_eval_results.json'
with open(results_path, 'w') as f:
    json.dump(serializable, f, indent=2)
print(f'Eval results saved → {results_path}')
print('\nAll outputs available in the Kaggle Output tab.')